# 🔎 Information Retrieval for RAG: BM25 vs Semantic vs Hybrid Search

### A hands-on Google Colab lesson

**Learning goals**
- Understand what BM25 does and why it is useful.
- Compare keyword search with semantic (embedding) search.
- Build a simple hybrid retriever.
- Understand why score normalization matters.
- See how these techniques fit into a RAG pipeline.

> **Teaching tip:** Run each section, inspect the output, and ask students to predict which documents should rank highest before revealing the results.

## 0. The problem: finding the right documents

Suppose we have a small collection of documents and a user asks a question.

Our job is to retrieve the most relevant documents before passing them to an LLM.

There are three approaches we'll compare:

1. **BM25** → matches important words.
2. **Semantic search** → matches meaning using embeddings.
3. **Hybrid search** → combines both signals.

This is one of the core ideas behind many RAG systems.

In [ ]:
# Install the libraries used in this notebook
!pip -q install rank-bm25 sentence-transformers

## 1. Our tiny document collection

We will deliberately keep the dataset small so that we can inspect every result.

In [ ]:
documents = [
    "Python is widely used for machine learning and data science.",
    "Transformers are neural networks commonly used for NLP tasks.",
    "BM25 is a keyword based information retrieval algorithm.",
    "Vector databases store embeddings for semantic search.",
    "RAG combines information retrieval with large language models.",
    "Redis can be used as a vector database for AI applications.",
    "PostgreSQL can store embeddings using the pgvector extension.",
    "Neural networks learn representations from large datasets.",
]

for i, doc in enumerate(documents):
    print(f"[{i}] {doc}")

[0] Python is widely used for machine learning and data science.
[1] Transformers are neural networks commonly used for NLP tasks.
[2] BM25 is a keyword based information retrieval algorithm.
[3] Vector databases store embeddings for semantic search.
[4] RAG combines information retrieval with large language models.
[5] Redis can be used as a vector database for AI applications.
[6] PostgreSQL can store embeddings using the pgvector extension.
[7] Neural networks learn representations from large datasets.


## 2. BM25: keyword-based retrieval

BM25 is a classic information-retrieval algorithm. It rewards documents that contain query terms, while accounting for:

- **Term frequency:** how often a query term appears.
- **Inverse document frequency (IDF):** rare terms are more informative.
- **Document length:** unusually long documents are normalized.
- **Term-frequency saturation:** repeating a word many times does not increase relevance forever.

A useful intuition is:

> **BM25 asks: “How strong is the lexical match between this query and this document?”**

In [ ]:
from rank_bm25 import BM25Okapi

# BM25 expects tokenized documents.
tokenized_documents = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_documents)

query = "How can I find relevant information for an LLM?"
tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

print("Query:", query)
print("\nBM25 scores:")
for i, score in enumerate(bm25_scores):
    print(f"[{i}] {score:.4f}  {documents[i]}")

Query: How can I find relevant information for an LLM?

BM25 scores:
[0] 0.0000  Python is widely used for machine learning and data science.
[1] 0.0000  Transformers are neural networks commonly used for NLP tasks.
[2] 0.9815  BM25 is a keyword based information retrieval algorithm.
[3] 0.0000  Vector databases store embeddings for semantic search.
[4] 0.9815  RAG combines information retrieval with large language models.
[5] 0.8438  Redis can be used as a vector database for AI applications.
[6] 0.9815  PostgreSQL can store embeddings using the pgvector extension.
[7] 0.0000  Neural networks learn representations from large datasets.


### Discussion question 💬

Look at the query and documents. Why might BM25 have difficulty finding the RAG document?

The query says **“LLM”**, while the document says **“large language models.”**
The meanings are related, but the exact tokens are different.

This is where semantic search can help.

## 3. Semantic search with embeddings

An embedding model converts text into a vector of numbers.

Texts with similar meanings tend to have vectors that are close together.

We'll use a small Sentence Transformers model that works well for a classroom demo.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

# With normalized vectors, dot product is equivalent to cosine similarity.
semantic_scores = document_embeddings @ query_embedding

print("Semantic scores:")
for i, score in enumerate(semantic_scores):
    print(f"[{i}] {score:.4f}  {documents[i]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic scores:
[0] 0.0719  Python is widely used for machine learning and data science.
[1] 0.0240  Transformers are neural networks commonly used for NLP tasks.
[2] 0.2520  BM25 is a keyword based information retrieval algorithm.
[3] 0.1614  Vector databases store embeddings for semantic search.
[4] 0.1341  RAG combines information retrieval with large language models.
[5] 0.1096  Redis can be used as a vector database for AI applications.
[6] 0.0620  PostgreSQL can store embeddings using the pgvector extension.
[7] 0.1235  Neural networks learn representations from large datasets.


## 4. Compare the rankings

Let's write a helper function so we can easily inspect the top results.

In [ ]:
def show_results(title, scores, top_k=5):
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    print("=" * 75)
    print(title)
    print("=" * 75)

    for rank, idx in enumerate(ranked_indices, start=1):
        print(f"{rank}. score={scores[idx]:.4f} | {documents[idx]}")


show_results("BM25", bm25_scores)
show_results("Semantic Search", semantic_scores)

BM25
1. score=0.9815 | PostgreSQL can store embeddings using the pgvector extension.
2. score=0.9815 | RAG combines information retrieval with large language models.
3. score=0.9815 | BM25 is a keyword based information retrieval algorithm.
4. score=0.8438 | Redis can be used as a vector database for AI applications.
5. score=0.0000 | Neural networks learn representations from large datasets.
Semantic Search
1. score=0.2520 | BM25 is a keyword based information retrieval algorithm.
2. score=0.1614 | Vector databases store embeddings for semantic search.
3. score=0.1341 | RAG combines information retrieval with large language models.
4. score=0.1235 | Neural networks learn representations from large datasets.
5. score=0.1096 | Redis can be used as a vector database for AI applications.


## 5. Hybrid search

BM25 and embeddings have complementary strengths:

| Method | Good at | Weakness |
|---|---|---|
| BM25 | Exact terms, product names, IDs, technical keywords | Can miss paraphrases |
| Semantic | Meaning, paraphrases, natural-language questions | Can miss exact/rare terms |
| Hybrid | Both lexical + semantic signals | More moving parts |

A simple hybrid formula is:

$$H(d) = \alpha B(d) + (1-\alpha)S(d)$$

where `B` is the normalized BM25 score and `S` is the normalized semantic score.

**Important:** raw BM25 and cosine-similarity scores are on different scales, so we normalize them before combining them.

In [ ]:
def min_max_normalize(scores):
    scores = np.asarray(scores, dtype=float)
    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)


bm25_normalized = min_max_normalize(bm25_scores)
semantic_normalized = min_max_normalize(semantic_scores)

# alpha = 1.0 means BM25 only.
# alpha = 0.0 means semantic search only.
alpha = 0.30

hybrid_scores = (
    alpha * bm25_normalized
    + (1 - alpha) * semantic_normalized
)

show_results(f"Hybrid Search (alpha={alpha})", hybrid_scores)

Hybrid Search (alpha=0.3)
1. score=1.0000 | BM25 is a keyword based information retrieval algorithm.
2. score=0.6382 | RAG combines information retrieval with large language models.
3. score=0.5209 | Redis can be used as a vector database for AI applications.
4. score=0.4220 | Vector databases store embeddings for semantic search.
5. score=0.4166 | PostgreSQL can store embeddings using the pgvector extension.


## 6. See how `alpha` changes the retriever

Try different values:

- `alpha = 1.0` → BM25 only
- `alpha = 0.7` → BM25-heavy hybrid
- `alpha = 0.5` → balanced
- `alpha = 0.3` → semantic-heavy
- `alpha = 0.0` → semantic only

This is a great place to pause and ask students: **“What should happen to the ranking as alpha changes?”**

In [ ]:
for alpha in [1.0, 0.7, 0.5, 0.3, 0.0]:
    hybrid = alpha * bm25_normalized + (1 - alpha) * semantic_normalized
    ranked = np.argsort(hybrid)[::-1][:3]

    print(f"\nalpha={alpha}")
    for rank, idx in enumerate(ranked, 1):
        print(f"  {rank}. {documents[idx]}")


alpha=1.0
  1. PostgreSQL can store embeddings using the pgvector extension.
  2. RAG combines information retrieval with large language models.
  3. BM25 is a keyword based information retrieval algorithm.

alpha=0.7
  1. BM25 is a keyword based information retrieval algorithm.
  2. RAG combines information retrieval with large language models.
  3. PostgreSQL can store embeddings using the pgvector extension.

alpha=0.5
  1. BM25 is a keyword based information retrieval algorithm.
  2. RAG combines information retrieval with large language models.
  3. Redis can be used as a vector database for AI applications.

alpha=0.3
  1. BM25 is a keyword based information retrieval algorithm.
  2. RAG combines information retrieval with large language models.
  3. Redis can be used as a vector database for AI applications.

alpha=0.0
  1. BM25 is a keyword based information retrieval algorithm.
  2. Vector databases store embeddings for semantic search.
  3. RAG combines information retrieval

## 7. A query where BM25 shines

Now let's search for a very specific technical term.

Ask students to predict the result before running the cell.

In [ ]:
query_2 = "PostgreSQL pgvector"

bm25_scores_2 = bm25.get_scores(query_2.lower().split())
query_embedding_2 = model.encode(query_2, normalize_embeddings=True)
semantic_scores_2 = document_embeddings @ query_embedding_2

show_results("BM25 — specific keyword query", bm25_scores_2, top_k=3)
show_results("Semantic — specific keyword query", semantic_scores_2, top_k=3)

BM25 — specific keyword query
1. score=3.3064 | PostgreSQL can store embeddings using the pgvector extension.
2. score=0.0000 | Neural networks learn representations from large datasets.
3. score=0.0000 | Redis can be used as a vector database for AI applications.
Semantic — specific keyword query
1. score=0.5894 | PostgreSQL can store embeddings using the pgvector extension.
2. score=0.3322 | Vector databases store embeddings for semantic search.
3. score=0.2153 | Redis can be used as a vector database for AI applications.


### Key observation

For exact technical terms such as `pgvector`, lexical retrieval can be extremely useful.

This is one reason production retrieval systems often **do not replace BM25 with embeddings completely**.

## 8. Where this fits in a RAG system

A simplified RAG pipeline looks like this:

```text
User question
      │
      ├───────────────┐
      ▼               ▼
    BM25          Embedding model
      │               │
      ▼               ▼
 Keyword results   Semantic results
      │               │
      └───────┬───────┘
              ▼
        Hybrid ranking
              │
              ▼
        Top-k chunks
              │
              ▼
             LLM
              │
              ▼
            Answer
```

In a real system, you may add a **reranker** after retrieval to improve the final ordering.

## 9. Challenge for students 🧪

Modify the document collection and test these queries:

1. `How do neural networks learn?`
2. `What is a vector database?`
3. `How does RAG use an LLM?`
4. `pgvector installation`
5. `Python machine learning`

For each query, compare BM25, semantic, and hybrid rankings.

### Bonus challenge
Add a document containing an exact product name or error code, then create a query using that identifier. Which retrieval method handles it better?

## 10. Takeaways 🎯

1. **BM25 is lexical retrieval.** It cares about words and their importance.
2. **Embeddings enable semantic retrieval.** They help match meaning even when wording differs.
3. **Neither method wins every query.** Their failure modes are different.
4. **Hybrid retrieval combines complementary signals.**
5. **Score normalization matters** when combining scores directly.
6. In production RAG, retrieval quality is often improved further with **reranking, metadata filtering, query rewriting, and evaluation**.

### The mental model

> **BM25 finds words. Embeddings find meaning. Hybrid search tries to get the best of both.**